# Butyrate Pathway Extraction from Human-GEM

**Obiettivo**: Estrarre reazioni pathway butirrato da Human-GEM per caninizzazione

**Output**: CSV con reazioni, geni (ENSG), GPR rules

---

## Workflow

1. Download Human-GEM model (SBML)
2. Keyword search reazioni correlate a butirrato
3. Estrazione GPR rules e ENSG IDs
4. Classificazione fasi pathway
5. Export CSV per Google Drive

---

## Background Biologico - Pathway Butirrato

**Dal cibo all'energia cellulare:**

1. **Produzione nel lume intestinale**
   - Le fibre alimentari vengono fermentate dai batteri intestinali (microbiota)
   - Questo processo produce butirrato, un acido grasso a catena corta (SCFA)

2. **Trasporto nella cellula**
   - Il butirrato attraversa la membrana cellulare del colonocita
   - Passa dal lume intestinale al citoplasma della cellula

3. **Attivazione metabolica**
   - Il butirrato viene convertito in butirril-CoA
   - Questa forma attivata può entrare nel ciclo di β-ossidazione

4. **β-Ossidazione**
   - Il butirril-CoA viene progressivamente degradato
   - Ogni ciclo rimuove 2 atomi di carbonio producendo acetil-CoA
   - L'acetil-CoA entra nel ciclo di Krebs

5. **Produzione energia**
   - Il ciclo di Krebs e la fosforilazione ossidativa producono ATP
   - Il butirrato fornisce ~70% dell'energia dei colonociti

---

---

## Block 1: Setup e Download Human-GEM

**Cosa fa**:
- Import librerie
- Download Human-GEM da GitHub (release latest)
- Load modello SBML

In [5]:
# Import
import cobra
import pandas as pd
import os
from pathlib import Path
import urllib.request

print(f"✓ COBRApy version: {cobra.__version__}")
print(f"✓ Pandas version: {pd.__version__}")

✓ COBRApy version: 0.29.1
✓ Pandas version: 2.2.3


In [2]:
# Download Human-GEM
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

model_path = data_dir / 'Human-GEM.xml'

if not model_path.exists():
    print("📥 Downloading Human-GEM (latest release)...")
    url = 'https://github.com/SysBioChalmers/Human-GEM/raw/main/model/Human-GEM.xml'
    urllib.request.urlretrieve(url, model_path)
    print(f"✓ Downloaded: {model_path.stat().st_size / 1e6:.1f} MB")
else:
    print(f"✓ Model già scaricato: {model_path}")

📥 Downloading Human-GEM (latest release)...
✓ Downloaded: 43.4 MB


In [6]:
# Load model
print("📂 Loading Human-GEM...")
model = cobra.io.read_sbml_model(str(model_path))

print(f"\n✓ Model loaded:")
print(f"  - Reactions: {len(model.reactions):,}")
print(f"  - Metabolites: {len(model.metabolites):,}")
print(f"  - Genes: {len(model.genes):,}")

# Anteprima prime 5 reazioni
print(f"\n📋 Anteprima prime 5 reazioni:")
print("-" * 80)
for i, rxn in enumerate(model.reactions[:5]):
    print(f"\n{i+1}. {rxn.id} - {rxn.name}")
    print(f"   Equation: {rxn.build_reaction_string()}")
    print(f"   GPR: {rxn.gene_reaction_rule[:60]}..." if len(rxn.gene_reaction_rule) > 60 else f"   GPR: {rxn.gene_reaction_rule}")
    print(f"   Bounds: [{rxn.lower_bound}, {rxn.upper_bound}]")

📂 Loading Human-GEM...

✓ Model loaded:
  - Reactions: 12,971
  - Metabolites: 8,455
  - Genes: 2,887

📋 Anteprima prime 5 reazioni:
--------------------------------------------------------------------------------

1. MAR03905 - ethanol:NAD+ oxidoreductase
   Equation: MAM01796c + MAM02552c --> MAM01249c + MAM02039c + MAM02553c
   GPR: ENSG00000147576 or ENSG00000172955 or ENSG00000180011 or ENS...
   Bounds: [0.0, 1000.0]

2. MAR03907 - Ethanol:NADP+ oxidoreductase
   Equation: MAM01796c + MAM02554c --> MAM01249c + MAM02039c + MAM02555c
   GPR: ENSG00000117448
   Bounds: [0.0, 1000.0]

3. MAR04097 - Acetate:CoA ligase (AMP-forming)
   Equation: MAM01252c + MAM01371c + MAM01597c --> MAM01261c + MAM01334c + MAM02759c
   GPR: ENSG00000131069
   Bounds: [0.0, 1000.0]

4. MAR04099 - Acetate:CoA ligase (AMP-forming)
   Equation: MAM01252m + MAM01371m + MAM01597m --> MAM01261m + MAM01334m + MAM02759m
   GPR: ENSG00000111058 or ENSG00000154930
   Bounds: [0.0, 1000.0]

5. MAR04108 - acetyl ad

---

## Block 2: Keyword Search Reazioni Butirrato

**Modalità:**
- Keyword search (default)
- Lista Reaction_ID specifica (opzionale)

In [17]:
# Keywords per ricerca (modalità default)
keywords = ['but', 'butyr']

# OPZIONALE: Lista specifica di Reaction_ID
# Se popolata, ignora keywords e usa questa lista
reaction_id_list = ["MAR03160"]  # Esempio: ["MAR09809", "MAR00742", "MAR04097"]

print(f"📋 Modalità: {'Lista Reaction_ID' if reaction_id_list else 'Keyword search'}")

📋 Modalità: Lista Reaction_ID


In [18]:
# Cerca reazioni con modalità Reaction_ID o Keyword
import re
found_reactions = []

if reaction_id_list:
    # Modalità: usa lista Reaction_ID
    print(f"🔍 Cerco {len(reaction_id_list)} Reaction_ID specifici...")
    for rxn_id in reaction_id_list:
        try:
            rxn = model.reactions.get_by_id(rxn_id)
            found_reactions.append(rxn)
        except KeyError:
            print(f"⚠️ Reaction_ID non trovato: {rxn_id}")
else:
    # Modalità: usa keyword
    print(f"🔍 Keyword search: {keywords}")
    for rxn in model.reactions:
        rxn_name_lower = rxn.name.lower()
        rxn_id_lower = rxn.id.lower()
        
        for keyword in keywords:
            if re.search(keyword, rxn_name_lower) or re.search(keyword, rxn_id_lower):
                found_reactions.append(rxn)
                break

# Rimuovi duplicati
found_reactions = list(set(found_reactions))

print(f"\n✓ Totale reazioni trovate: {len(found_reactions)}")

🔍 Cerco 1 Reaction_ID specifici...

✓ Totale reazioni trovate: 1


In [19]:
# Anteprima prime 10 reazioni trovate
print("\n📋 ANTEPRIMA PRIME 10 REAZIONI TROVATE")
print("=" * 80)

for i, rxn in enumerate(found_reactions[:10]):
    print(f"\n{i+1}. {rxn.id} - {rxn.name}")
    
    # Equation troncata
    equation = rxn.build_reaction_string()
    equation_display = equation[:70] + "..." if len(equation) > 70 else equation
    print(f"   Equation: {equation_display}")
    
    # GPR troncato
    gpr_display = rxn.gene_reaction_rule[:50] + "..." if len(rxn.gene_reaction_rule) > 50 else rxn.gene_reaction_rule
    print(f"   GPR: {gpr_display}")


📋 ANTEPRIMA PRIME 10 REAZIONI TROVATE

1. MAR03160 - butanoyl-CoA:acetyl-CoA C-butanoyltransferase
   Equation: MAM00882m + MAM01597m --> MAM01261m + MAM01412m
   GPR: ENSG00000084754 and ENSG00000138029 and ENSG000001...


In [12]:
# Export Block 2 results to CSV
import pandas as pd
from pathlib import Path

# Crea output dir
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# Costruisci DataFrame
block2_data = []
for rxn in found_reactions:
    block2_data.append({
        'Reaction_ID': rxn.id,
        'Name': rxn.name,
        'Equation': rxn.build_reaction_string(),
        'GPR_Original': rxn.gene_reaction_rule,
        'Lower_Bound': rxn.lower_bound,
        'Upper_Bound': rxn.upper_bound
    })

df_block2 = pd.DataFrame(block2_data)

# Export
csv_path = output_dir / 'block2_keyword_search_results.csv'
df_block2.to_csv(csv_path, index=False)

print(f"\n✓ Esportato: {csv_path}")
print(f"  Righe: {len(df_block2)}")
print(f"  Dimensione: {csv_path.stat().st_size / 1024:.1f} KB")


✓ Esportato: output/block2_keyword_search_results.csv
  Righe: 91
  Dimensione: 11.4 KB


**✅ Test Block 2**: Verifica che siano trovate ~50-60 reazioni totali

---

In [ ]:
# Funzione estrazione compartimenti
def extract_compartments(rxn):
    """Estrae compartimenti da equazione reazione"""
    compartments = set()
    for metabolite in rxn.metabolites:
        # Compartimento è ultimo carattere dopo '[' in ID
        if '[' in metabolite.id:
            comp = metabolite.id.split('[')[-1].rstrip(']')
            compartments.add(comp)
    return sorted(list(compartments))

# Test
print(f"\nTest compartimenti:")
print(f"  {test_rxn.id}: {extract_compartments(test_rxn)}")

In [ ]:
# Costruisci dataset completo
data_rows = []

for rxn in found_reactions:
    ensg_list = extract_ensg_from_gpr(rxn)
    
    data_rows.append({
        'Reaction_ID': rxn.id,
        'Name': rxn.name,
        'Equation': rxn.build_reaction_string(),
        'Compartments': ', '.join(extract_compartments(rxn)),
        'ENSG_List': '; '.join(ensg_list) if ensg_list else '',
        'GPR_Original': rxn.gene_reaction_rule,
        'Lower_Bound': rxn.lower_bound,
        'Upper_Bound': rxn.upper_bound
    })

# Crea DataFrame
df = pd.DataFrame(data_rows)

print(f"\n✓ Dataset costruito: {len(df)} reazioni × {len(df.columns)} colonne")

**✅ Test Block 3**: Verifica che df contenga tutte le reazioni con colonne complete

---

## Block 4: Anteprima Dati

**Cosa fa**:
- Mostra statistiche dataset
- Anteprima prime righe
- Conta geni unici

In [ ]:
# Statistiche generali
print("📊 STATISTICHE DATASET")
print("=" * 50)
print(f"Totale reazioni: {len(df)}")
print(f"Reazioni con geni: {(df['ENSG_List'] != '').sum()}")
print(f"Reazioni senza geni: {(df['ENSG_List'] == '').sum()}")

# Conta ENSG unici
all_ensg = set()
for ensg_str in df['ENSG_List']:
    if ensg_str:
        all_ensg.update([e.strip() for e in ensg_str.split(';')])

print(f"\nGeni unici (ENSG): {len(all_ensg)}")

In [ ]:
# Anteprima prime 10 reazioni
print("\n📋 ANTEPRIMA PRIME 10 REAZIONI")
print("=" * 50)

# Mostra colonne principali
display_cols = ['Reaction_ID', 'Name', 'ENSG_List']
df[display_cols].head(10)

In [ ]:
# Esempi reazioni
print("\n🔬 ESEMPI REAZIONI")
print("=" * 50)

for i, row in df.head(3).iterrows():
    print(f"\n{i+1}. {row['Reaction_ID']}")
    print(f"   Nome: {row['Name']}")
    print(f"   ENSG: {row['ENSG_List'][:50]}..." if len(row['ENSG_List']) > 50 else f"   ENSG: {row['ENSG_List']}")
    print(f"   Compartments: {row['Compartments']}")

**✅ Test Block 4**: Verifica statistiche coerenti e anteprima leggibile

---

## Block 5: Export CSV per Google Drive

**Cosa fa**:
- Salva dataset in CSV
- Crea file separato con solo lista ENSG unici (per BioMart)
- Mostra path file generati

In [ ]:
# Crea output directory
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# Export reazioni complete
csv_path = output_dir / 'butyrate_reactions.csv'
df.to_csv(csv_path, index=False)

print(f"✓ Esportato: {csv_path}")
print(f"  Righe: {len(df)}")
print(f"  Dimensione: {csv_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Export lista ENSG unici (per BioMart)
ensg_df = pd.DataFrame({
    'ENSG_ID': sorted(list(all_ensg))
})

ensg_csv_path = output_dir / 'ensg_list_for_biomart.csv'
ensg_df.to_csv(ensg_csv_path, index=False)

print(f"\n✓ Esportato: {ensg_csv_path}")
print(f"  ENSG unici: {len(ensg_df)}")

In [ ]:
# Summary finale
print("\n" + "=" * 50)
print("✅ EXPORT COMPLETATO")
print("=" * 50)
print(f"\nFile generati:")
print(f"1. {csv_path}")
print(f"   → Importa in Google Drive/Sheets")
print(f"\n2. {ensg_csv_path}")
print(f"   → Usa per BioMart (Human→Dog orthologs)")
print(f"\nProssimi step:")
print(f"  1. Upload CSV su Google Drive")
print(f"  2. BioMart: mappa ENSG umani → ENSCAFG canini")
print(f"  3. Sostituisci GPR con geni canini")

**✅ Test Block 5**: Verifica file CSV esistono e hanno contenuto corretto

---

## Fine Notebook

**Output pronti per Google Drive** ✓